# Preparación del conjunto de datos


## Dataset
[Osteosarcoma-Tumor-Assesment](https://www.cancerimagingarchive.net/collection/osteosarcoma-tumor-assessment/)

> Leavey, P., Sengupta, A., Rakheja, D., Daescu, O., Arunachalam, H. B., & Mishra, R. (2019). Osteosarcoma data from UT Southwestern/UT Dallas for Viable and Necrotic Tumor Assessment (Osteosarcoma-Tumor-Assessment) [Data set]. The Cancer Imaging Archive. https://doi.org/10.7937/tcia.2019.bvhjhdas

## Limpieza de datos

El conjunto de 1144 imágenes viene con un CSV que describe las clases, y un total de 20 carpetas con imagenes. Lo cambiaremos a un formato más tradicional de train/test sets.

In [1]:
import zipfile

# Specify the path to your ZIP file
zip_file_path = '../data/PKG - Osteosarcoma Tumor Assessment.zip'

# Specify the target directory where you want to extract the contents
extraction_path = '../data'

try:
    # Open the ZIP file in read mode ('r')
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # Extract all contents to the specified directory
        zip_ref.extractall(extraction_path)
    print(f"Successfully extracted '{zip_file_path}' to '{extraction_path}'")
except zipfile.BadZipFile:
    print(f"Error: '{zip_file_path}' is not a valid ZIP file.")
except FileNotFoundError:
    print(f"Error: The file '{zip_file_path}' was not found.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully extracted '../data/PKG - Osteosarcoma Tumor Assessment.zip' to '../data'


In [2]:
import pandas as pd

# Leer el csv
csv_file_path = "../data/PKG - Osteosarcoma Tumor Assessment/ML_Features_1144.csv"
df = pd.read_csv(csv_file_path)

# Corrige errores en los archivos
df['image.name'] = df['image.name'].str.replace(' - ', '-', regex=False)
df['image.name'] = df['image.name'].str.replace(' ', '-', regex=False)

# Reemplazar etiquetas incorrectas por las correctas
df["classification"] = df["classification"].replace({
    "viable: non-viable": "Viable-Tumor",
    "Viable": "Viable-Tumor",
    "Non-Viable-Tumor": "Necrotic-Tumor"
})

image_class_lookup = dict(zip(df["image.name"], df["classification"]))

# Show first 5 items from the lookup table
print("First 5 entries from the lookup dictionary:")
for k in list(image_class_lookup.keys())[:5]:
    print(k, ":", image_class_lookup[k])

First 5 entries from the lookup dictionary:
Case-3-A10-10547-25283 : Non-Tumor
Case-3-A10-10566-40206 : Non-Tumor
Case-3-A10-13444-20223 : Non-Tumor
Case-3-A10-14507-37285 : Non-Tumor
Case-3-A10-14726-26052 : Non-Tumor


In [3]:
import os
import shutil

root_directory = '../data/PKG - Osteosarcoma Tumor Assessment/'

# Directorio temporal para concentrar las imágenes.
tmp_dir = '../data/tmp/'
os.makedirs(tmp_dir, exist_ok=True)

moved_count = 0

print(f"Buscando imágenes en '{root_directory}' y subdirectorios (hasta 2 niveles)...\n")

for dirpath, dirnames, filenames in os.walk(root_directory):
    # Calculate the current depth relative to the root_directory
    current_depth = dirpath[len(root_directory):].count(os.sep)

    # Debug print: Show current directory and its depth
    # print(f"DEBUG: Evaluando directorio: {dirpath}, Profundidad: {current_depth}")

    # Only process directories up to 2 levels deep (0, 1, 2)
    if current_depth > 2:
        del dirnames[:] # Don't recurse into deeper directories
        # print(f"DEBUG: Saltando directorio (demasiado profundo): {dirpath}")
        continue

    for filename in filenames:
        if filename.lower().endswith(('.jpg', '.jpeg')):
            source_path = os.path.join(dirpath, filename)
            destination_path = os.path.join(tmp_dir, filename)

            # Handle potential filename conflicts by renaming if necessary
            base, ext = os.path.splitext(filename)
            counter = 1
            while os.path.exists(destination_path):
                destination_path = os.path.join(tmp_dir, f"{base}_{counter}{ext}")
                counter += 1

            try:
                shutil.move(source_path, destination_path)
                moved_count += 1
                # print(f"Movido: {source_path} -> {destination_path}") # Uncomment for detailed logs
            except Exception as e:
                print(f"Error al mover '{source_path}' a '{destination_path}': {e}")

print(f"\nSe han movido {moved_count} imágenes a '{tmp_dir}'")
shutil.rmtree(root_directory)


Buscando imágenes en '../data/PKG - Osteosarcoma Tumor Assessment/' y subdirectorios (hasta 2 niveles)...


Se han movido 1144 imágenes a '../data/tmp/'


In [4]:
import os
import pandas as pd

output_directory = "../data/osteosarcoma2019"
os.makedirs(output_directory, exist_ok=True)

image_filenames = [f for f in os.listdir(tmp_dir) if f.lower().endswith(('.jpg', '.jpeg'))]

for filename in image_filenames:
    # Quitar la extensión para buscar en el diccionario
    name_without_ext = os.path.splitext(filename)[0]

    # Buscar la clasificación
    classification = image_class_lookup.get(name_without_ext)

    if classification is None:
        print(f"[WARNING] '{name_without_ext}' no está en el CSV. Se omite.")
        continue

    # Crear carpeta de la clase si no existe
    class_folder = os.path.join(output_directory, classification)
    os.makedirs(class_folder, exist_ok=True)

    # Rutas origen : destino
    src = os.path.join(tmp_dir, filename)
    dst = os.path.join(class_folder, filename)

    # Mover el archivo
    shutil.move(src, dst)

os.removedirs(tmp_dir)
print("Archivos movidos.")

for nombre in os.listdir(output_directory):
    subdir = os.path.join(output_directory, nombre)
    if os.path.isdir(subdir):
        count = sum(1 for f in os.listdir(subdir)
                    if os.path.isfile(os.path.join(subdir, f)))
        print(f"{nombre}: {count} archivos")

Archivos movidos.
Non-Tumor: 536 archivos
Necrotic-Tumor: 263 archivos
Viable-Tumor: 345 archivos


En este punto tenemos una carpeta con las 3 clases.

### Dividir en subsets

In [5]:
%run ../scripts/prepare_data.py --src ../data/osteosarcoma2019 --dst ../data/splits --seed 42

El directorio destino ya existe: ../data/splits
Fuente : /home/asdromundo/Documentos/classification/data/osteosarcoma2019
Destino: /home/asdromundo/Documentos/classification/data/splits
Semilla: 42

[1] Recopilando imágenes...
  Non-Tumor: 536 imágenes encontradas
  Viable-Tumor: 345 imágenes encontradas
  Necrotic-Tumor: 263 imágenes encontradas

  Total: 1144 imágenes en 3 clases

[2] Separando test set fijo (20% del total)...
    → test: 230 imágenes copiadas
       Non-Tumor: 108
       Viable-Tumor: 69
       Necrotic-Tumor: 53

  Pool de entrenamiento: 914 | Test fijo: 230

[3] Generando subsets de entrenamiento...

  [full] fracción=1.0
    → full/train: 730 imágenes copiadas
       Non-Tumor: 342
       Viable-Tumor: 220
       Necrotic-Tumor: 168
    → full/val: 184 imágenes copiadas
       Non-Tumor: 86
       Viable-Tumor: 56
       Necrotic-Tumor: 42

  [subset_25] fracción=0.25
    → subset_25/train: 181 imágenes copiadas
       Non-Tumor: 85
       Viable-Tumor: 55
      